[← 08a - Changing Data, Part 1](<08a-changing-data-part-1.ipynb>) · [Course Overview](<00-course-overview.ipynb>)

# 08b - Changing Data, Part 2: CREATE TABLE, constraints, and ALTER TABLE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/08b-changing-data-part-2.ipynb)

`08a` changed rows inside rules somebody else wrote, and ended with Chidi deleted out from under his own order because nothing said he couldn't be. This notebook writes the rules. It reads the setup cell's `CREATE TABLE` line by line, adds the constraints that stop bad rows at the door, declares the foreign key that would have saved order 105, and uses `ALTER TABLE` to change a table that's already full of rows, along with the one `INSERT` habit that keeps those changes from breaking your scripts.

*New here? The [Course Overview](<00-course-overview.ipynb>) explains how to run these notebooks. Viewing this on GitHub shows a preview with saved results; it won't execute anything.*

> **By the end of this notebook you'll be able to:** read and write a `CREATE TABLE` with types and constraints, use `NOT NULL`, `PRIMARY KEY`, `UNIQUE`, `CHECK`, and `DEFAULT` to keep bad rows out, declare a foreign key and switch it on in SQLite, change a live table with `ALTER TABLE`, and explain why an `INSERT` without a column list is a script waiting to break.

> **Read the full lesson:** this notebook is the hands-on companion to [Lesson 8 of the All About Data & More curriculum](LESSON_8_URL), which walks through the reasoning in more depth.

## 0. Set up

Run the cell below once each time you open the notebook. After it, any cell that starts with `%%sql` runs SQLite against a small lesson database that lives in memory, so there's nothing to install, and any cell that starts with `%%tsql` runs against SQL Server and AdventureWorks, which only section 9 uses and which has its own one-step setup there.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


## 1. The same two tables

Run the setup once. If you've been working through `08a` in the same session, run `%sql_reset` in a new cell first, so the tables start exactly as they were.

In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


## 2. CREATE TABLE, line by line

`CREATE TABLE customers (...)` makes an empty table, and each line inside the parentheses defines one column as a name, a **type**, and optionally some rules:

- `customer_id INTEGER PRIMARY KEY` says the column holds whole numbers and no two rows may share a value, which is what lets a join trust that `customer_id = 2` means exactly one customer.
- `customer_name TEXT NOT NULL` says the column holds text and can never be empty.
- `credit_limit INTEGER NOT NULL` says the limit is a whole number that must always be there.

Those rules aren't comments; the database checks them on every write. Each of these cells is meant to fail. The first adds a customer with no name:

In [3]:
%%sql --expect-error
INSERT INTO customers (
    customer_id,
    customer_name,
    credit_limit
)
VALUES (5, NULL, 150);

Error (this cell is meant to fail): NOT NULL constraint failed: customers.customer_name


The second reuses an order number that already exists:

In [4]:
%%sql --expect-error
INSERT INTO orders (
    order_id,
    customer_id,
    amount
)
VALUES (101, 4, 50);

Error (this cell is meant to fail): UNIQUE constraint failed: orders.order_id


In [5]:
%%sql
SELECT
    (SELECT COUNT(*) FROM customers) AS customers,
    (SELECT COUNT(*) FROM orders) AS orders;

customers,orders
4,5


Still four customers and five orders: neither bad row got in. A **constraint** is a rule the database enforces for you, so a bad row gets stopped at the door instead of discovered three reports later.

## 3. More rules: CHECK, UNIQUE, and DEFAULT

Three more constraints cover most of what you'll need. Here's a `payments` table that uses them, recording how each order was paid:

In [6]:
%%sql
CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    amount INTEGER NOT NULL CHECK (amount > 0),
    method TEXT NOT NULL DEFAULT 'card' CHECK (method IN ('card', 'cash', 'transfer')),
    reference TEXT UNIQUE
);

- `CHECK (amount > 0)` rejects any row where the condition is false, so a payment can't be zero or negative.
- `CHECK (method IN (...))` limits a column to a list of allowed values.
- `DEFAULT 'card'` fills in a value when an `INSERT` leaves the column out.
- `UNIQUE` on `reference` means no two payments can share a reference, like a primary key that isn't the row's identity.

A good payment, leaving out `method` so the default applies:

In [7]:
%%sql
INSERT INTO payments (
    payment_id,
    order_id,
    amount,
    reference
)
VALUES (1, 101, 40, 'PAY-001');

1 row inserted


Now three that break a rule each. These cells are meant to fail: a negative amount, a payment method that isn't on the list, and a reference that's already been used.

In [8]:
%%sql --expect-error
INSERT INTO payments (
    payment_id,
    order_id,
    amount,
    method,
    reference
)
VALUES (2, 102, -5, 'cash', 'PAY-002');

Error (this cell is meant to fail): CHECK constraint failed: amount > 0


In [9]:
%%sql --expect-error
INSERT INTO payments (
    payment_id,
    order_id,
    amount,
    method,
    reference
)
VALUES (3, 103, 80, 'cheque', 'PAY-003');

Error (this cell is meant to fail): CHECK constraint failed: method IN ('card', 'cash', 'transfer


In [10]:
%%sql --expect-error
INSERT INTO payments (
    payment_id,
    order_id,
    amount,
    method,
    reference
)
VALUES (4, 104, 90, 'cash', 'PAY-001');

Error (this cell is meant to fail): UNIQUE constraint failed: payments.reference


### Edge case: UNIQUE and missing values

Two payments with no reference at all:

In [11]:
%%sql
INSERT INTO payments (
    payment_id,
    order_id,
    amount,
    method,
    reference
)
VALUES
    (5, 105, 140, 'cash', NULL),
    (6, 102, 60, 'transfer', NULL);

2 rows inserted


In [12]:
%%sql
SELECT
    payment_id,
    order_id,
    amount,
    method,
    reference
FROM payments;

payment_id,order_id,amount,method,reference
1,101,40,card,PAY-001
5,105,140,cash,NULL
6,102,60,transfer,NULL


Both went in. `UNIQUE` stops two rows sharing a *value*, and `NULL` isn't a value, it's the absence of one, so SQLite, PostgreSQL, and MySQL all allow any number of `NULL`s in a `UNIQUE` column. SQL Server is the exception: its `UNIQUE` constraint allows only one `NULL`. If "no reference" should never happen, the column needs `NOT NULL` as well as `UNIQUE`. Notice too that payment 1 shows `card`, filled in by the `DEFAULT`.

## 4. Foreign keys: the rule that would have saved Chidi's order

A **foreign key** declares that a column must match an existing value in another table's key. For `orders`, the second column becomes:

```sql
customer_id INTEGER NOT NULL REFERENCES customers (customer_id),
```

And here's the SQLite-specific catch. The [SQLite foreign key documentation](https://www.sqlite.org/foreignkeys.html) says foreign key constraints "are disabled by default (for backwards compatibility), so must be enabled separately for each database connection", using `PRAGMA foreign_keys = ON;`. Switch it on, then build a pair of tables that declare the foreign key:

In [13]:
%%sql
PRAGMA foreign_keys = ON;

In [14]:
%%sql
CREATE TABLE fk_customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE fk_orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES fk_customers (customer_id),
    amount INTEGER NOT NULL
);
INSERT INTO fk_customers
SELECT
    customer_id,
    customer_name,
    credit_limit
FROM customers;
INSERT INTO fk_orders
SELECT
    order_id,
    customer_id,
    amount
FROM orders;

4 rows inserted
5 rows inserted


Now try `08a`'s delete of Chidi, which left order 105 behind. This cell is meant to fail:

In [15]:
%%sql --expect-error
DELETE FROM fk_customers
WHERE customer_id = 3;

Error (this cell is meant to fail): FOREIGN KEY constraint failed


Refused. So is an order for a customer who doesn't exist (also meant to fail):

In [16]:
%%sql --expect-error
INSERT INTO fk_orders (
    order_id,
    customer_id,
    amount
)
VALUES (106, 9, 50);

Error (this cell is meant to fail): FOREIGN KEY constraint failed


Deleting Dami, who has no orders, still works, because nothing points at her (inside a transaction, so she comes back):

In [17]:
%%sql
BEGIN;

DELETE FROM fk_customers
WHERE customer_id = 4;

ROLLBACK;

1 row deleted


<p align="center">
  <img src="graphics/08a_orphaned_order.png" width="640" alt="Without an enforced foreign key, deleting Chidi leaves order 105 behind and reports disagree; with REFERENCES and PRAGMA foreign_keys = ON, the delete is rejected and every report agrees">
</p>

### Edge case: the switch that quietly does nothing

The pragma has one more trap: run inside an open transaction, it's silently ignored. Watch it report `0` (off) even after being set to `ON`:

In [18]:
%%sql
PRAGMA foreign_keys = OFF;
BEGIN;
PRAGMA foreign_keys = ON;
PRAGMA foreign_keys;
ROLLBACK;

foreign_keys
0


In [19]:
%%sql
PRAGMA foreign_keys = ON;
PRAGMA foreign_keys;

foreign_keys
1


Run the pragma before any `BEGIN`, and in SQLite, at the start of every connection, since it doesn't persist. Every other major engine, SQL Server included, enforces foreign keys always, as section 9 shows.

## 5. ALTER TABLE: changing the structure

`ALTER TABLE` changes a table that already has rows in it. Adding a column is the most common use:

In [20]:
%%sql
ALTER TABLE customers
ADD COLUMN email TEXT;

In [21]:
%%sql
SELECT
    customer_id,
    customer_name,
    credit_limit,
    email
FROM customers;

customer_id,customer_name,credit_limit,email
1,Ada,100,NULL
2,Bola,200,NULL
3,Chidi,300,NULL
4,Dami,200,NULL


Every existing customer now has an `email` column, and every one of them holds `NULL`, because the table can't know four addresses it was never given. If the new column must be `NOT NULL`, the existing rows need a value, so you must give a default. This cell is meant to fail:

In [22]:
%%sql --expect-error
ALTER TABLE orders
ADD COLUMN status TEXT NOT NULL;

Error (this cell is meant to fail): Cannot add a NOT NULL column with default value NULL


With a `DEFAULT`, it works, and every existing row takes the default (inside a transaction, so `orders` keeps its usual three columns):

In [23]:
%%sql
BEGIN;

ALTER TABLE orders
ADD COLUMN status TEXT NOT NULL DEFAULT 'paid';

SELECT
    order_id,
    amount,
    status
FROM orders;

ROLLBACK;

order_id,amount,status
101,40,paid
102,60,paid
103,80,paid
104,90,paid
105,140,paid


SQLite's [ALTER TABLE](https://www.sqlite.org/lang_altertable.html) is deliberately small: it can rename a table, rename a column, add a column, or drop one, and that's the whole list. Changing a column's type or adding a constraint to an existing table means building a new table and copying the rows across, which is why the foreign key in section 4 went into a `CREATE TABLE` rather than an `ALTER`. SQL Server can do both directly, as section 9 shows.

<p align="center">
  <img src="graphics/08b_alter_table.png" width="720" alt="ADD COLUMN and DROP COLUMN work in SQLite and SQL Server; ALTER COLUMN and ADD CONSTRAINT work in SQL Server, while SQLite needs the table rebuilt">
</p>

## 6. Common mistake: INSERT without a column list

Now, with `email` in place, run the setup cell's style of insert for Efe:

In [24]:
%%sql --expect-error
INSERT INTO customers VALUES (5, 'Efe', 150);

Error (this cell is meant to fail): table customers has 4 columns but 3 values were supplied


"4 columns but 3 values". The statement didn't change; the table did. The column-list version still works, leaving Efe's `email` as `NULL`, because it says which three columns its three values belong to:

In [25]:
%%sql
BEGIN;

INSERT INTO customers (
    customer_id,
    customer_name,
    credit_limit
)
VALUES (5, 'Efe', 150);

SELECT
    customer_id,
    customer_name,
    credit_limit,
    email
FROM customers
WHERE customer_id = 5;

ROLLBACK;

1 row inserted


customer_id,customer_name,credit_limit,email
5,Efe,150,NULL


That's the loud version of this mistake, and it's the lucky one. Drop the column so `customers` is back to its usual shape (`DROP COLUMN` needs SQLite 3.35.0 or later):

In [26]:
%%sql
ALTER TABLE customers
DROP COLUMN email;

The quiet version is getting the values in the wrong order. Here's Efe's insert with the name and limit swapped, inside a transaction:

In [27]:
%%sql
BEGIN;

INSERT INTO customers VALUES (5, 150, 'Efe');

SELECT
    customer_id,
    customer_name,
    credit_limit
FROM customers
WHERE customer_id = 5;

ROLLBACK;

1 row inserted


customer_id,customer_name,credit_limit
5,150,Efe


SQLite accepted it: a customer named `'150'` with a credit limit of `'Efe'`. The [datatype documentation](https://www.sqlite.org/datatype3.html) explains why: SQLite ties a type to each *value* rather than to the column, unlike "every SQL database engine other than SQLite, as far as we know", so an `INTEGER` column will hold text that doesn't look like a number. A table created with the `STRICT` keyword (SQLite 3.37.0 and later) enforces column types like other engines do. Create one, then try the same insert; the second cell is meant to fail:

In [28]:
%%sql
CREATE TABLE strict_customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
) STRICT;

In [29]:
%%sql --expect-error
INSERT INTO strict_customers VALUES (5, 150, 'Efe');

Error (this cell is meant to fail): cannot store TEXT value in INTEGER column strict_customers.credit_limit


Either way, naming the columns makes the order you meant visible on the page, and that's the habit that protects you in every engine.

## 7. Every write is a decision

Reading data asks a question, while writing data changes the answer for everyone who asks after you. `CREATE TABLE` sets the rules, `INSERT`, `UPDATE`, and `DELETE` change rows within those rules, `ALTER TABLE` changes the rules themselves, and the database enforces only the rules you actually declared (and, for a foreign key in SQLite, switched on). The habits that carry over to every engine are small: name your columns, declare the rules you're relying on, and test them with a row that should fail.

## 8. Try it yourself

Write each statement in a new cell before looking at the [solutions notebook](<solutions/08b-changing-data-part-2-solutions.ipynb>), predicting the result first.

1. **Design a table.** Write a `CREATE TABLE products` with a whole-number ID as its primary key, a name that's required and can't repeat, a price that must be greater than 0, and an `in_stock` flag that defaults to 1. Then prove each rule works with an `INSERT` that should fail.
2. **Which inserts get in?** Using `payments` from section 3, predict which of these succeed before running them one at a time: (a) payment 7 for order 103, amount 80, no method, reference `'PAY-007'`; (b) payment 8 for order 104, amount 0, method `'cash'`, reference `'PAY-008'`; (c) payment 9 for order 105, amount 140, method `'card'`, reference `'PAY-001'`; (d) payment 10 for order 104, amount 90, method `'transfer'`, reference `NULL`.
3. **Add a column safely.** Add a `signup_year` column to `customers` that every existing customer gets as 2026, and that can never be empty. Run it inside a transaction and roll it back.
4. **Stretch, on AdventureWorks.** SQL Server records every foreign key in a system view called `sys.foreign_keys`. Find which table the most foreign keys point at. (`OBJECT_NAME(referenced_object_id)` turns an ID into a table name.) Save this one for after section 9.

## 9. Scale it up: rules in SQL Server

SQLite let several things through in this notebook that most engines refuse. This section checks each one on **SQL Server**, using a temporary `#customers` table so AdventureWorks itself is never changed, and then looks at the foreign keys AdventureWorks declares.

### Set up AdventureWorks

SQL Server is a separate program from this notebook, so it needs one setup step before the `%%tsql` cells can run:

- **In Google Colab**, the cell below installs SQL Server and AdventureWorks inside your Colab session for you. The first run takes a few minutes and prints each step as it goes. Colab gives you a fresh machine every session, so run it again whenever you come back to the notebook later.
- **On your own machine**, it checks that SQL Server is reachable and tells you what's missing if it isn't; the [Course Overview](<00-course-overview.ipynb>) covers installing SQL Server and AdventureWorks locally.

If you'd rather not set anything up, every `%%tsql` cell below keeps its saved result, so you can read the section without running it.

In [30]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


### A typed table

SQL Server writes text types as `VARCHAR(n)`, where `n` is the maximum length, and whole numbers as `INT`:

In [31]:
%%tsql
CREATE TABLE #customers (
    customer_id INT PRIMARY KEY,
    customer_name VARCHAR(50) NOT NULL,
    credit_limit INT NOT NULL
);
INSERT INTO #customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);

4 rows inserted


### The wrong-order insert, refused

Section 6's quiet mistake, which SQLite accepted. This cell is meant to fail:

In [32]:
%%tsql --expect-error
INSERT INTO #customers VALUES (5, 150, 'Efe');

Error (this cell is meant to fail): Conversion failed when converting the varchar value 'Efe' to data type int.


SQL Server's columns have fixed types, so `'Efe'` can't become a credit limit. (Look closely and you'll see `150` was accepted as a name, since a number converts to text without complaint; only the second swap was caught. Naming the columns is still the real protection.)

### Adding a required column

The same rule as SQLite: a `NOT NULL` column needs a default for the rows already there. This cell is meant to fail:

In [33]:
%%tsql --expect-error
ALTER TABLE #customers
ADD status VARCHAR(10) NOT NULL;

Error (this cell is meant to fail): ALTER TABLE only allows columns to be added that can contain nulls, or have a DEFAULT definition specified, or the column being added is an identity or timestamp column, or alternatively if none of the previous conditions are satisfied the table must be empty to allow addition of this column. Column 'status' cannot be added to non-empty table '#customers' because it does not satisfy these conditions.


In [34]:
%%tsql
ALTER TABLE #customers
ADD status VARCHAR(10) NOT NULL DEFAULT 'active';
GO
SELECT
    customer_id,
    customer_name,
    credit_limit,
    status
FROM #customers;

customer_id,customer_name,credit_limit,status
1,Ada,100,active
2,Bola,200,active
3,Chidi,300,active
4,Dami,200,active


SQL Server writes `ADD`, without the word `COLUMN`. The `GO` line splits the cell into two batches, because SQL Server won't let a statement use a column added earlier in the same batch.

### What SQLite's ALTER TABLE can't do

Changing a column's type, here letting credit limits have cents:

In [35]:
%%tsql
ALTER TABLE #customers
ALTER COLUMN credit_limit DECIMAL(10, 2) NOT NULL;
GO
SELECT
    customer_id,
    customer_name,
    credit_limit
FROM #customers;

customer_id,customer_name,credit_limit
1,Ada,100.00
2,Bola,200.00
3,Chidi,300.00
4,Dami,200.00


And adding a constraint to a table that already has rows. SQL Server checks the new rule against every existing row first, so a cap of 250 fails because Chidi's limit is 300. This cell is meant to fail:

In [36]:
%%tsql --expect-error
ALTER TABLE #customers
ADD CONSTRAINT credit_limit_cap CHECK (credit_limit <= 250);

Error (this cell is meant to fail): The ALTER TABLE statement conflicted with the CHECK constraint "credit_limit_cap". The conflict occurred in database "tempdb", table "dbo.#customers", column 'credit_limit'.

A rule every row already obeys goes in, and from then on it guards every write. The second cell is meant to fail:

In [37]:
%%tsql
ALTER TABLE #customers
ADD CONSTRAINT credit_limit_positive CHECK (credit_limit > 0);

In [38]:
%%tsql --expect-error
UPDATE #customers
SET credit_limit = -5
WHERE customer_id = 1;

Error (this cell is meant to fail): The UPDATE statement conflicted with the CHECK constraint "credit_limit_positive". The conflict occurred in database "tempdb", table "dbo.#customers", column 'credit_limit'.


### A real foreign key at work

AdventureWorks declares 90 foreign keys, and SQL Server always enforces them, with no switch to turn on. Customer 29825 placed twelve orders. This delete is meant to fail, and because it fails, nothing is changed:

In [39]:
%%tsql --expect-error
DELETE FROM Sales.Customer
WHERE CustomerID = 29825;

Error (this cell is meant to fail): The DELETE statement conflicted with the REFERENCE constraint "FK_SalesOrderHeader_Customer_CustomerID". The conflict occurred in database "AdventureWorks2022", table "Sales.SalesOrderHeader", column 'CustomerID'.


That's `08a`'s orphan, prevented: the constraint `FK_SalesOrderHeader_Customer_CustomerID` refuses to let an order lose its customer. Here's how many foreign keys the database declares:

In [40]:
%%tsql
SELECT COUNT(*) AS foreign_keys
FROM sys.foreign_keys;

foreign_keys
90


## Before you trust a table's rules

- Read the `CREATE TABLE`: the rules it declares are the only ones the database enforces.
- Use `NOT NULL` for anything that must always be there, and `UNIQUE` plus `NOT NULL` for anything that must be there and never repeat.
- In SQLite, check `PRAGMA foreign_keys;` returns 1, and run it before any `BEGIN`.
- Give a new `NOT NULL` column a `DEFAULT`, or it can't be added to a table with rows.
- Name the columns in every `INSERT`, so adding a column later can't break it or scramble it.
- Test each rule with a row that should fail.

## Go deeper

- **[CREATE TABLE](https://www.sqlite.org/lang_createtable.html)**, **[Foreign Key Support](https://www.sqlite.org/foreignkeys.html)**, **[ALTER TABLE](https://www.sqlite.org/lang_altertable.html)**, and **[STRICT Tables](https://www.sqlite.org/stricttables.html)**, SQLite. Everything this notebook used, including the full list of what `ALTER TABLE` can and can't do and the recipe for rebuilding a table.
- **[Datatypes In SQLite](https://www.sqlite.org/datatype3.html)**. Why SQLite stored a name in a number column, and how it differs from every other engine.
- **[CREATE TABLE](https://learn.microsoft.com/en-us/sql/t-sql/statements/create-table-transact-sql)**, **[ALTER TABLE](https://learn.microsoft.com/en-us/sql/t-sql/statements/alter-table-transact-sql)**, and **[Primary and foreign key constraints](https://learn.microsoft.com/en-us/sql/relational-databases/tables/primary-and-foreign-key-constraints)**, Microsoft Learn. The SQL Server references behind section 9.
- **[PostgreSQL: Constraints](https://www.postgresql.org/docs/current/ddl-constraints.html)**. The clearest single explanation of every constraint type, including what foreign keys can do on delete.
- **Practice:** [SQLBolt](https://sqlbolt.com/lesson/creating_tables) lessons 16 to 18 cover creating, altering, and dropping tables in the browser.

For interview questions on this topic with model answers, see the [study guide](<study-guides/08b-changing-data-part-2.md>).

## What's next

You can now read and write every row in these tables, and declare the rules they follow. `09 - Thinking in Metrics` steps back from syntax altogether: the harder question is how a business question like "who are our best customers?" turns into a query, and why two correct queries can give two different answers.

---

[← 08a - Changing Data, Part 1](<08a-changing-data-part-1.ipynb>) · [Course Overview](<00-course-overview.ipynb>) · [09 - Thinking in Metrics →](<09-thinking-in-metrics.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).